# Road Sign Detection with YOLO

Trains a multi-class UK road sign detector in Google Colab using a [Roboflow](https://roboflow.com) dataset and [Ultralytics](https://github.com/ultralytics/ultralytics) YOLO.

**Steps:** install deps &rarr; download dataset &rarr; train &rarr; validate &rarr; test inference &rarr; export `best.pt`.

See `training/README.md` for how to find/prepare a dataset and get a Roboflow API key.

> Runtime: Colab menu &rarr; Runtime &rarr; Change runtime type &rarr; **GPU** (T4 is fine).

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 2. Download the dataset from Roboflow

Get a free API key from your Roboflow account (Settings &rarr; API Keys) and either paste it below or, better, store it as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left sidebar) so it isn't saved in the notebook.

Replace `WORKSPACE`, `PROJECT`, and `VERSION` with the values from the dataset's Roboflow page (Universe search: "UK road signs", "UK traffic signs", "GTSRB UK", etc.). The snippet is generated for you on the dataset's "Download" tab — pick the **YOLOv8** export format.

In [ ]:
import os
from roboflow import Roboflow

try:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    api_key = os.environ.get("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

WORKSPACE = "your-workspace"
PROJECT = "your-project"
VERSION = 1

rf = Roboflow(api_key=api_key)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(dataset.location)

## 3. Train

Starts from the pretrained `yolo11n.pt` (nano) checkpoint for a quick baseline. Swap in `yolo11s.pt`/`yolo11m.pt` for higher accuracy once the pipeline works end to end.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="uk-road-signs",
    name="train",
)

## 4. Validate

Runs the best checkpoint from training against the validation split and reports mAP50 / mAP50-95, precision, and recall.

In [ ]:
best_weights = results.save_dir / "weights" / "best.pt"
print(f"Best weights: {best_weights}")

best_model = YOLO(best_weights)
metrics = best_model.val(data=f"{dataset.location}/data.yaml")

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 5. Test inference

Runs the trained model on a few images from the test split and previews the annotated output, including per-detection confidence scores.

In [ ]:
import glob
from IPython.display import Image, display

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:5]

for img_path in test_images:
    preds = best_model.predict(img_path, conf=0.25, save=True)
    for r in preds:
        for box in r.boxes:
            cls_name = best_model.names[int(box.cls)]
            conf = float(box.conf)
            print(f"{img_path}: {cls_name} ({conf:.2f})")
        display(Image(filename=str(r.save_dir / os.path.basename(img_path))))

## 6. Export weights

Downloads `best.pt` so it can be dropped into `app/backend/weights/` for the FastAPI service. You can also export to ONNX for faster/smaller inference.

In [ ]:
from google.colab import files

files.download(str(best_weights))

# Optional: also export an ONNX version
# best_model.export(format="onnx")